In [4]:
sample_text = """
John Smith applied for a Python Developer job.
Email: john@gmail.com
Experience: 2 years.
Expected salary: 45000 INR.
"""

print(sample_text)


John Smith applied for a Python Developer job.
Email: john@gmail.com
Experience: 2 years.
Expected salary: 45000 INR.



In [5]:
!pip install -q google-genai pydantic

In [10]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini AI connected successfully!")

Gemini AI connected successfully!


In [11]:
class Candidate(BaseModel):
    name: str
    email: str
    job_role: str
    experience_years: int = Field(ge=0)
    expected_salary: float = Field(gt=0)

print("Candidate data model created!")

print(json.dumps(
    Candidate.model_json_schema(),
    indent=4
))

Candidate data model created!
{
    "properties": {
        "name": {
            "title": "Name",
            "type": "string"
        },
        "email": {
            "title": "Email",
            "type": "string"
        },
        "job_role": {
            "title": "Job Role",
            "type": "string"
        },
        "experience_years": {
            "minimum": 0,
            "title": "Experience Years",
            "type": "integer"
        },
        "expected_salary": {
            "exclusiveMinimum": 0,
            "title": "Expected Salary",
            "type": "number"
        }
    },
    "required": [
        "name",
        "email",
        "job_role",
        "experience_years",
        "expected_salary"
    ],
    "title": "Candidate",
    "type": "object"
}


In [22]:
def extract_candidate_details(text):

    prompt = f"""
    You are an expert recruitment data extraction assistant.

    Extract the candidate information from the text.

    Important:
    - Extract the actual salary if mentioned.
    - Extract experience as a number.
    - Do not invent information.
    - If salary is missing, use 0.
    - If experience is missing, use 0.
    - If name is missing, use Unknown.
    - If email is missing, use unknown@example.com.
    - If job role is missing, use Not specified.

    Candidate text:
    {text}
    """

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": Candidate
        }
    )

    result = Candidate.model_validate_json(response.text)

    return result

print("AI extraction function updated successfully!")

AI extraction function updated successfully!


In [24]:
sample_text = """
John Smith applied for a Python Developer job.
Email: john@gmail.com
Experience: 2 years.
Expected salary: 45000 INR.
"""

result = extract_candidate_details(sample_text)

print("\n========== AI EXTRACTION RESULT ==========\n")

print(result.model_dump_json(indent=4))


========== AI EXTRACTION RESULT ==========

{
    "name": "John Smith",
    "email": "john@gmail.com",
    "job_role": "Python Developer",
    "experience_years": 2,
    "expected_salary": 45000.0
}
